# Retrieving parts of Molecules

> ### In this tutorial we will cover:
> - how to retrieve individual atoms, residues, and chains
> - how to use item getting [] syntax

When building or manipulating molecules we need to interact with its components, namely the atoms, residues, and chains that form its data hierarchy. The `Molecule` class offers convenient methods to retrieve these entities based on a variety of inputs. 

## Retrieving individual entities 

We can retrieve single atoms etc. by using the following methods

#### Atoms
Using the `get_atom` method, which accepts as input either
- atom id / name (e.g. C1, HO4, etc.)
- atom serials (e.g. 1, 2, 3, etc.)
- elements (e.g. H, C, AU)

#### Residues
Using the `get_residue` method, which accepts as input either
- residue id / name (e.g. GLC, LIG, HIS, etc.)
- residue serials (e.g. 1, 3, 3, etc.)

#### Chains
Using the `get_chain` method, which accepts as input
- chain id / name (e.g. A, B, etc.)

#### Models
Using the `get_model` method, which accepts as input
- model id / name (e.g. 0, 1, 2, etc.)

The methods accept an additional parameter `"by"` which can be used to specify the type of input we want to search by, but by default the methods try to automatically infer the correct search, by determining the datatype that is provided. 

> The only exception is atom-level element-based retrieval which usually requires to set `by="element"`

In [1]:
import buildamol as bam
mol = bam.molecule("glucose")

In [2]:
# retrieve the C1 atom by name
c1 = mol.get_atom("C1")
print(c1)

# retrieve the atom with serial 5
atom5 = mol.get_atom(5)
print(atom5)

# retrieve the first (and only) residue in the molecule
residue = mol.get_residue(1)
print(residue)

Atom(C1, 7)
Atom(O5, 5)
Residue(UNK, 1)


> A note on *serial numbers*. As by PDB convention any serial number index (atoms, residues, etc.) start from `1` rather than 0! You can change this by using `mol.reindex(start_atomid=0, ...)` if you must.

## Retrieving multiple entities

The `get_{}` methods return invidual entities. However, these methods have plural-equivalents (named `get_atoms`, `get_residues`, etc.) taht will return **all** matching entities in a list. 

In [3]:
# retrieve atoms C1, C2, and C3
atoms = mol.get_atoms(["C1", "C2", "C3"])
print(atoms)

# retrieve all oxygen atoms
oxygens = mol.get_atoms("O", by="element")
print(oxygens)

[Atom(C1, 7), Atom(C2, 8), Atom(C3, 9)]
[Atom(O1, 1), Atom(O2, 2), Atom(O3, 3), Atom(O4, 4), Atom(O5, 5), Atom(O6, 6)]


#### Filtering

`get_atoms` and `get_residues` also support a `filter` argument which accepts a callable that can be used to filter the results on the fly. 

In [4]:
from buildamol.structural import constraints_v2 as cc

# modify O1 to be a nitrogen
mol.get_atom("O1").set_element("N")

# retrieve all carbons that neighbor a Nitrogen
neighbors_oxygen = cc.neighbors_any("N") # a constraint function constructor...
carbons_neighboring_oxygen = mol.get_atoms("C", by="element", filter=neighbors_oxygen) 
print(carbons_neighboring_oxygen)

[Atom(C3, 9), Atom(C5, 11)]


#### Plain-form iterators

If no arguments are provided to the `get_{}s` methods then they will behave like the default generators from Biopython. 

In [5]:
atoms = mol.get_atoms()
print(atoms)

<generator object Model.get_atoms at 0x13dbfa810>


#### Plain-form lists

The `Molecule` also offers access to premade lists (rather than generators) to access its sub-entities. There are two versions available, the sorted and unsorted ones. 
Sorted means the entities are sorted based on their id/name. Unsorted means they are left exactly in the order they are yielded by the plain form generators. 
This works for atoms, residues, and chains.

In [6]:
atoms_sorted = mol.atoms
atoms_unsorted = mol.atoms_
print("Sorted\t\tUnsorted")
for a, b in zip(atoms_sorted, atoms_unsorted):
    print(a, b, sep="\t")

Sorted		Unsorted
Atom(H1, 13)	Atom(N1, 1)
Atom(H2, 14)	Atom(O2, 2)
Atom(H3, 15)	Atom(O3, 3)
Atom(H4, 16)	Atom(O4, 4)
Atom(H5, 17)	Atom(O5, 5)
Atom(H6, 18)	Atom(O6, 6)
Atom(H7, 19)	Atom(C1, 7)
Atom(H8, 20)	Atom(C2, 8)
Atom(H9, 21)	Atom(C3, 9)
Atom(H10, 22)	Atom(C4, 10)
Atom(H11, 23)	Atom(C5, 11)
Atom(H12, 24)	Atom(C6, 12)
Atom(C1, 7)	Atom(H1, 13)
Atom(C2, 8)	Atom(H2, 14)
Atom(C3, 9)	Atom(H3, 15)
Atom(C4, 10)	Atom(H4, 16)
Atom(C5, 11)	Atom(H5, 17)
Atom(C6, 12)	Atom(H6, 18)
Atom(N1, 1)	Atom(H7, 19)
Atom(O2, 2)	Atom(H8, 20)
Atom(O3, 3)	Atom(H9, 21)
Atom(O4, 4)	Atom(H10, 22)
Atom(O5, 5)	Atom(H11, 23)
Atom(O6, 6)	Atom(H12, 24)


## Item getting

Biopython entities support itemgetting with the `[]` operator, therefore BuildAMol `Residues`, `Chains`, and `Models` do it too from inheritence. However, the toplevel `Molecule` does **not** by default. The reason being that - as we have seen with the getter methods above, the way we can / might want to retrieve entites will differ from case to case, meaning what is convenient for one person or molecule may not be convenient for another. Therefore, item getting is by default disabled. However, it can be activated using the `set_getitem` method which allows the user to specify the logic they want to use for item getting. 

Possible item getting logics are:

- atom-serial (retrieve atom by serial number)
- atom-index (retrieve atom by index in the unsorted list of atoms)
- atom-id (retrieve atom by id/name, will return the first match)
- residue-serial (retrieve residue by serial number)
- residue-index (retrieve residue by index in the unsorted list of residues)
- residue-id (retrieve residue by id/name, will return the first match)
- chain-id (retrieve chain by id/name, will return the first match)
- chain-index (retrieve chain by index in the unsorted list of chains)
- model-id (retrieve model by id/name, will return the first match)
- model-index (retrieve model by index in the list of models)
  
If none of these meet the needs, a custom function that accepts `f(mol, index)` as only arguments can also be set. 

In [ ]:
# enable item getting via indexing (proper pythonic 0-based indexing)
mol.set_getitem("atom-index")
print(mol[0])

# enable item getting via atom names
mol.set_getitem("atom-id")
print(mol["C1"])

Atom(N1, 1)
Atom(C1, 7)


> To enable item getting globally one can set the class variable `Molecule.default_getitem_method` to any of these values to enable all instantiated Molecules to support itemgetting by default.

That's it for this tutorial. We have seen that we can obtain both individual and bulk elements of molecules via a handful of methods or item getting. Thanks for checking out this tutorial and best of luck with your project using BuildAMol!